# NLP session 2 - Full fine-tuning of a small BERT classifier

**Goal:** adapt a pretrained encoder to binary sentiment classification, evaluate before/after and reload the result.
**Classroom time:** 45 minutes including explanations and discussion.

This notebook is self-contained. Upload it to [Colab](https://colab.research.google.com/) and execute cells in order.
It uses public model/data repositories with no Hugging Face token or paid API.
A Google account and Internet are required to execute in Colab. CPU works, with a reduced teaching mode; GPU is optional.
Start installation early. Colab runtime speed, download speed and GPU availability vary.

**Full fine-tuning:** every parameter of the encoder and classification head is trainable. No LoRA, adapters or frozen layers.
The encoder is already pretrained; the new two-label head starts randomly. We do not retrain notebook 01's DistilBERT.


## 1. Install the course versions


In [ ]:
%pip -q install "transformers==4.57.1" "datasets==4.3.0" "accelerate==1.11.0"


## 2. Imports and runtime

Run the next cell before any other imports. If different library versions were already imported, restart the session after installation and run from here.


In [ ]:
import os
# Run this before importing transformers: this lesson uses PyTorch only.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import time
from pathlib import Path
from collections import Counter
from importlib.metadata import version
import numpy as np
import torch
from datasets import load_dataset
from transformers import set_seed

SEED = 42
set_seed(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 2))
print({p: version(p) for p in ["torch", "transformers", "datasets", "accelerate"]})
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")


## 3. Choose the teaching mode

- `auto`: standard on GPU, reduced on CPU.
- `standard`: 1,024 training examples, 2 epochs (64 updates at batch size 32).
- `reduced`: 256 training examples, 1 epoch (8 updates at batch size 32).

Both modes use 256 validation examples and at most 64 tokens per example. The reduced run demonstrates the procedure; its accuracy improvement may be small or absent.
Use the default for a first run. If there is time on CPU, you may choose `standard` **before creating the datasets and model**.


In [ ]:
MODE = "auto"  # "auto", "standard" or "reduced"
assert MODE in {"auto", "standard", "reduced"}
reduced = MODE == "reduced" or (MODE == "auto" and not torch.cuda.is_available())
N_TRAIN = 256 if reduced else 1024
N_VALID = 256
EPOCHS = 1 if reduced else 2
BATCH_SIZE = 32
MAX_LENGTH = 64
LEARNING_RATE = 2e-4
MODEL_ID = "google/bert_uncased_L-2_H-128_A-2"
print({"mode": "reduced" if reduced else "standard", "train": N_TRAIN,
       "validation": N_VALID, "epochs": EPOCHS, "batch": BATCH_SIZE})


## 4. Build separate balanced subsets

We preserve the official split boundaries. We select equal counts of each class and remove exact duplicate texts after lowercasing and whitespace normalization.
The helper is supplied: concentrate on **where the data come from** and **what is held out**.
Similar but non-identical texts may remain; this simple check does not prove the absence of every form of overlap.
The training split contains sentences and phrase fragments. The validation split contains sentences.

**Question A:** Why must validation examples stay out of parameter updates?


In [ ]:
dataset = load_dataset("stanfordnlp/sst2")

def normalize(text):
    return " ".join(text.lower().split())

def balanced_subset(split, n, forbidden=None):
    assert n % 2 == 0
    forbidden = set() if forbidden is None else set(forbidden)
    shuffled = split.shuffle(seed=SEED)
    counts = Counter()
    seen = set()
    indices = []
    for i, example in enumerate(shuffled):
        key = normalize(example["sentence"])
        label = int(example["label"])
        if key in forbidden or key in seen or counts[label] >= n // 2:
            continue
        seen.add(key)
        indices.append(i)
        counts[label] += 1
        if len(indices) == n:
            break
    if len(indices) != n:
        raise ValueError("Not enough unique examples for a balanced subset.")
    return shuffled.select(indices).shuffle(seed=SEED)

validation_raw = balanced_subset(dataset["validation"], N_VALID)
validation_texts = {normalize(t) for t in validation_raw["sentence"]}
train_raw = balanced_subset(dataset["train"], N_TRAIN, forbidden=validation_texts)
assert not ({normalize(t) for t in train_raw["sentence"]} & validation_texts)
print("Training labels:", Counter(train_raw["label"]))
print("Validation labels:", Counter(validation_raw["label"]))
print("One training example:", train_raw[0])
print("One validation example:", validation_raw[0])


## 5. Tokenize and inspect one example

Use the checkpoint's own tokenizer. Keep the raw text, including negation and stop words.
Truncation limits length. The collator adds only the padding required by each batch.

**Question B:** What do `input_ids` and `attention_mask` contain? Why can token count differ from word count?


In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
def tokenize(batch):
    return tokenizer(batch["sentence"], truncation=True, max_length=MAX_LENGTH)

train_data = train_raw.map(tokenize, batched=True, remove_columns=["sentence", "idx"])
validation_data = validation_raw.map(tokenize, batched=True, remove_columns=["sentence", "idx"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(train_data[0])
print(tokenizer.convert_ids_to_tokens(train_data[0]["input_ids"]))
print("Raw length, before truncation:", len(tokenizer(train_raw[0]["sentence"])["input_ids"]))


## 6. Initialize the pretrained encoder and new head

The warning about **newly initialized classifier weights** is expected: the pretrained base checkpoint has no trained sentiment head.
The evaluation before training therefore concerns an encoder plus a **random head**, not a competent ready-made sentiment classifier.

We also check that all parameters are trainable and keep a copy of encoder embeddings to verify a real update later.
The fixed seed improves reproducibility, but different hardware/library kernels can still produce different results.


In [ ]:
from transformers import AutoModelForSequenceClassification

set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
    use_safetensors=True,
)
total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert all(p.requires_grad for p in model.parameters())
assert trainable_parameters == total_parameters
encoder_before = model.bert.embeddings.word_embeddings.weight.detach().cpu().clone()
print(f"Trainable parameters: {trainable_parameters:,} / {total_parameters:,}")


## 7. Read the training settings

`Trainer` handles batches, forward passes, loss, backpropagation and optimizer steps.
The learning rate controls update size. An epoch is one pass through the training subset.
We use AdamW, FP32 and dynamic padding. No external experiment logger or Hub upload runs.

**Question C:** How many optimizer updates are expected in your selected mode?


In [ ]:
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": float((predictions == labels).mean())}

training_args = TrainingArguments(
    output_dir="sst2-training-output",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=4,
    logging_first_step=True,
    report_to="none",
    push_to_hub=False,
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0,
    dataloader_pin_memory=torch.cuda.is_available(),
    optim="adamw_torch",
    fp16=False,
    bf16=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=validation_data,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("Expected optimizer steps:", int(np.ceil(N_TRAIN / BATCH_SIZE) * EPOCHS))


## 8. Measure before training

The balanced validation subset gives a majority-label reference of 50%.
Keep the measured random-head result as the before-training comparison. Its accuracy is not a measure of all pretrained encoder knowledge.


In [ ]:
gold = np.array(validation_raw["label"])
majority_accuracy = float((gold == 0).mean())
before = trainer.evaluate()
print("Majority-label baseline:", majority_accuracy)
print("Before fine-tuning:", before)


## 9. Run full fine-tuning

Run this cell **once** for the first experiment. Rerunning it continues from already updated weights, so restart from model initialization for a fresh before/after experiment.
Observe the loss, but wait for validation before drawing conclusions.


In [ ]:
started = time.perf_counter()
train_result = trainer.train()
training_seconds = time.perf_counter() - started
encoder_after = model.bert.embeddings.word_embeddings.weight.detach().cpu()
max_encoder_change = float((encoder_after - encoder_before).abs().max())
assert max_encoder_change > 0, "Encoder weights did not change. Check training."
print(f"Training time: {training_seconds:.1f} seconds")
print(f"Maximum embedding weight change: {max_encoder_change:.6f}")


## 10. Evaluate on exactly the same validation examples

**Question D:** What changed in accuracy, in percentage points? Did the result improve over 50%?
A gain is not guaranteed, particularly with the reduced run. Report the actual result.
Reusing validation to tune many settings would require an additional untouched test set for a final estimate.


In [ ]:
after = trainer.evaluate()
prediction_output = trainer.predict(validation_data)
predicted = prediction_output.predictions.argmax(axis=-1)
delta_points = 100 * (after["eval_accuracy"] - before["eval_accuracy"])
print(f"Majority reference: {majority_accuracy:.3f}")
print(f"Before: {before['eval_accuracy']:.3f}")
print(f"After:  {after['eval_accuracy']:.3f}")
print(f"Change: {delta_points:+.1f} percentage points")

confusion = np.zeros((2, 2), dtype=int)
for target, prediction in zip(gold, predicted):
    confusion[int(target), int(prediction)] += 1
print("Confusion matrix: rows = true labels, columns = predicted labels")
print("Label order: 0 NEGATIVE, 1 POSITIVE")
print(confusion)
errors = np.flatnonzero(predicted != gold)
for i in errors[:3]:
    i = int(i)
    print("\nTEXT:", validation_raw[i]["sentence"])
    print("TRUE:", int(gold[i]), "PREDICTED:", int(predicted[i]))


## 11. Save the model and verify reloading

We save locally, with the tokenizer and label mappings. This does not publish anything to the Hub.
The new pipeline loads the model from the saved directory.
The assertion checks that saving/reloading preserves predicted labels on two probes, not that these labels are correct.


In [ ]:
from transformers import pipeline

SAVE_DIR = "sst2-small-finetuned"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
probes = ["The film was excellent.", "The film was boring."]

model.eval()
inputs = tokenizer(probes, padding=True, truncation=True, max_length=MAX_LENGTH,
                   return_tensors="pt").to(model.device)
with torch.no_grad():
    original_ids = model(**inputs).logits.argmax(dim=-1).cpu().tolist()
original_labels = [model.config.id2label[i] for i in original_ids]

reloaded = pipeline("text-classification", model=SAVE_DIR, tokenizer=SAVE_DIR,
                    framework="pt", device=-1)
reloaded_outputs = reloaded(probes, truncation=True, max_length=MAX_LENGTH)
assert original_labels == [r["label"] for r in reloaded_outputs]
for text, result in zip(probes, reloaded_outputs):
    print(text, "=>", result)


## 12. Export the model and report

Complete the observation below. Run this cell, then download **sst2-small-finetuned.zip** from the Files panel on the left.
The archive includes model weights, tokenizer and the JSON experiment report.
Colab files can disappear when the runtime is reset, so download your work before leaving.


In [ ]:
import shutil

report = {
    "model": MODEL_ID,
    "dataset": "stanfordnlp/sst2",
    "mode": "reduced" if reduced else "standard",
    "seed": SEED,
    "n_train": N_TRAIN,
    "n_validation": N_VALID,
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "total_parameters": total_parameters,
    "trainable_parameters": trainable_parameters,
    "majority_accuracy": majority_accuracy,
    "before_accuracy": before["eval_accuracy"],
    "after_accuracy": after["eval_accuracy"],
    "change_percentage_points": delta_points,
    "training_seconds": training_seconds,
    "maximum_embedding_weight_change": max_encoder_change,
    "confusion_matrix": confusion.tolist(),
    "observation": "Explain one error or limitation here.",
    "versions": {p: version(p) for p in ["torch", "transformers", "datasets", "accelerate"]},
}
Path(SAVE_DIR, "experiment-report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
archive = shutil.make_archive(SAVE_DIR, "zip", root_dir=SAVE_DIR)
print("Download this file from the Files panel:", archive)


## Submission and discussion

Submit the initial and final accuracy, the majority reference, your hardware/mode, and one inspected error.
Explain which parameters changed and why validation differs from training loss.
Do not compare notebook 01's DistilBERT and this BERT-Tiny as if their difference were a controlled effect of your fine-tuning.

Optional after class: change **one** setting in a fresh run, log it, and discuss the uncertainty. A larger validation score after many attempts is not an independent test result.

**References**
- [Google BERT miniatures](https://huggingface.co/google/bert_uncased_L-2_H-128_A-2)
- [SST-2](https://huggingface.co/datasets/stanfordnlp/sst2)
- [Text classification tutorial](https://huggingface.co/docs/transformers/v4.57.1/en/tasks/sequence_classification)
- [Trainer](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer)
